# Detección de Luces en Semáforos Peatonales

**Proyecto:** SafeCross — Asistencia para personas con discapacidad visual al cruzar calles  
**Modelo:** YOLOv8m (fine-tuning sobre dataset personalizado)  
**Clases detectadas:**
- `0` — red light (luz roja)
- `1` — yellow light (luz amarilla)
- `2` — green light (luz verde)

Este notebook integra los pasos completos del pipeline:
1. Preparación de anotaciones (XML → YOLO)
2. Entrenamiento con fine-tuning
3. Evaluación / Testing

## 1. Preparación de Archivos

Convierte las anotaciones desde formato XML (Pascal VOC) al formato de texto
utilizado por YOLO (bounding boxes normalizadas).

**Estructura esperada antes de ejecutar:**
```
dataset/
└── annotations/
    └── xml/   ← archivos .xml aquí
```

**Resultado:**
```
dataset/
└── annotations/
    └── output/  ← archivos .txt generados
```

In [ ]:
import os
import xml.etree.ElementTree as ET

# Clases del modelo (índice = class_id en YOLO)
CLASSES = ["red light", "yellow light", "green light"]


def convert(size, box):
    """Convierte coordenadas absolutas a formato normalizado YOLO."""
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0 - 1
    y = (box[2] + box[3]) / 2.0 - 1
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)


def convert_annotation(xml_file, output_dir, classes):
    """Parsea XML y escribe el .txt equivalente para YOLO."""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    size_node = root.find("size")
    img_w = int(size_node.find("width").text)
    img_h = int(size_node.find("height").text)

    out_path = os.path.join(
        output_dir,
        os.path.splitext(os.path.basename(xml_file))[0] + ".txt",
    )
    with open(out_path, "w") as out_file:
        for obj in root.iter("object"):
            cls = obj.find("name").text
            if cls not in classes:
                continue
            cls_id = classes.index(cls)
            xmlbox = obj.find("bndbox")
            b = (
                float(xmlbox.find("xmin").text),
                float(xmlbox.find("xmax").text),
                float(xmlbox.find("ymin").text),
                float(xmlbox.find("ymax").text),
            )
            bb = convert((img_w, img_h), b)
            out_file.write(f"{cls_id} " + " ".join(str(v) for v in bb) + "\n")


# Rutas
current_path = os.getcwd()
xml_dir = os.path.join(current_path, "dataset", "annotations", "xml")
output_dir = os.path.join(current_path, "dataset", "annotations", "output")
os.makedirs(output_dir, exist_ok=True)

xml_files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
print(f"Convirtiendo {len(xml_files)} archivos XML...")

for xml_file in xml_files:
    convert_annotation(os.path.join(xml_dir, xml_file), output_dir, CLASSES)

print(f"Conversión completada. Salida en: {output_dir}")

## 2. Entrenamiento

Fine-tuning del modelo YOLOv8m pre-entrenado para adaptarlo a la detección
de luces de semáforos peatonales.

**Requisitos:**
- `yolov8m.pt` en el directorio actual (se descarga automáticamente si no existe)
- `TrafficLight.yaml` configurado con las rutas del dataset
- GPU disponible (recomendado)

**Salida:**
- Modelo guardado en `yolov8m_TrafficLight.pt`
- Métricas y checkpoints en `runs/detect/yolov8m_traffic_light/`

In [ ]:
from ultralytics import YOLO
import os

current_path = os.getcwd()

# Cargar el modelo base pre-entrenado YOLOv8m
model = YOLO("yolov8m.pt")

# Fine-tuning
results = model.train(
    data=os.path.join(current_path, "TrafficLight.yaml"),
    epochs=80,
    imgsz=640,
    batch=16,
    name="yolov8m_traffic_light",
    patience=20,
    device=0,
)

# Guardar el modelo entrenado
model.save("yolov8m_TrafficLight.pt")
print("Modelo guardado: yolov8m_TrafficLight.pt")

## 3. Evaluación / Testing

Ejecuta inferencia con el mejor checkpoint del entrenamiento sobre las
imágenes de prueba.

**Requisitos:**
- `best.pt` disponible (mejor checkpoint generado en el entrenamiento)
- Imágenes de prueba en `dataset/images/test/`

**Salida:**
- Imágenes anotadas en `runs/detect/predict/`
- Detecciones en texto en `runs/detect/predict/labels/`

In [ ]:
from ultralytics import YOLO
import os

current_path = os.getcwd()

# Cargar el mejor checkpoint
model = YOLO("best.pt")

# Ejecutar predicción
results = model.predict(
    source=os.path.join(current_path, "dataset", "images", "test"),
    save=True,
    save_txt=True,
)

print("Predicción completada. Ver resultados en: runs/detect/predict/")